# Final Project: Camera + Flight Pre-Tests

This notebook checks the analog camera setup (the camera module + Skydroid 5.8G dual receiver) before collecting imitation-learning trajectories with `keyboard_control_IL.py`:

0. **Hardware setup**: mount the camera and plug in the receiver.
1. **Find the receiver**: scan camera indices.
2. **Live view**: open the feed, then hold the receiver's left button so it locks onto your camera.
3. **Crop check**: check that the default crop (80 rows) removes the receiver's overlay.
4. **Record and play back** -- a 5 second handheld clip.
5. **Flight test** -- take off, hover, land. Arms the drone first (required on the Crazyflie 2.1 Brushless).
6. **Fly while recording** -- checks that the video link and radio work together and that the drone flies with the camera on.

Run this in the `mae345` environment:

```
conda activate mae345
cd <this folder>
jupyter lab
```

**macOS camera permission:** the first time OpenCV opens a camera, macOS asks to allow camera access for the app running Python (Terminal, iTerm, or VS Code). If you denied it, or nothing opens, enable it under *System Settings → Privacy & Security → Camera* and restart that app.

## Part 0: Hardware Setup

### Instructions for adding camera module for crazyflie

Remove the battery retainer, the optical flow module, and the pins attached to your board. The camera module will act as the battery holder and also keep the optical flow module attached. **Carefully** push the camera module where the battery retainer was until the pins stick out of the botton of the main body, ensure that it is facing foward on the drone and that there is enough room for the battery to slip between the main board and the camera module. Replace the optical flow board on these pins and position it as it was before. A picture is provided below (`photos/drone_with_camera.jpg`), and a youtube video of assembly is accessible [here](https://youtu.be/DavXkm0TiyY).

> **Note:** these instructions were written for the brushed Crazyflie 2.1. On the 2.1 Brushless, check that the camera sits facing forward, the battery still slides in, and the Flow deck is re-seated flat underneath, since Part 5 depends on the Flow deck for height and position.

![Drone with camera](photos/drone_with_camera.jpg)

### Connecting the receiver

The Skydroid receiver plugs in over USB and shows up as an ordinary webcam, so plain `cv2.VideoCapture(index)` reads it. You don't need to set a channel on the camera. The receiver finds it by scanning:

1. **Plug the receiver into the Mac.** Plug it straight into the Mac if you can. Through a hub it may show up on USB without appearing as a camera.
2. **Run Part 1** to find its camera index (the 640x480 one).
3. **Power on the drone** (this powers the camera) and **run Part 2** to open the live view. It will most likely show static.
4. **Hold down the receiver's left button** for a few seconds, then let go. It sweeps every frequency and locks onto the strongest signal. The drone's view appears in the window.
5. **Wave a hand in front of your drone's camera** to check that the picture is yours. Press `q` to close the window.

The receiver keeps its channel after you unplug it, so you only redo step 4 if the picture is lost.

**If the picture doesn't react to your hand,** the receiver locked onto another group's camera. Keep your drone right next to the receiver and scan again (step 4), or ask the other groups to switch their cameras off for a few seconds while you scan. If that still doesn't work, put the cameras on different channels: press the button on the camera to change `CH` (red LED, 1-8 left to right), and hold it for 2 s to change `BAND` (blue LED, 1-5). Then scan again. Frequencies (MHz):

| Channel | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| Band 1   | 5865 | 5845 | 5825 | 5805 | 5785 | 5765 | 5745 | 5725 |
| Band 2   | 5733 | 5752 | 5771 | 5790 | 5809 | 5828 | 5847 | 5866 |
| Band 3   | 5705 | 5685 | 5665 | 5665 | 5885 | 5905 | 5905 | 5905 |
| Band 4   | 5740 | 5760 | 5780 | 5800 | 5820 | 5840 | 5860 | 5880 |
| Band 5   | 5658 | 5695 | 5732 | 5769 | 5806 | 5843 | 5880 | 5917 |

If a cv2 window stops responding, restart the kernel (Kernel → Restart).

## Part 1: Find the Receiver's Camera Index

On a Mac the numbering varies: the built-in FaceTime camera and a nearby iPhone (Continuity Camera) also take indices, and the receiver may come before or after them. The cell below tries the first few indices and shows one frame from each. The receiver is the **640x480** one, showing static or the drone's view (the built-in camera is 1920x1080). Set `camera` to its index in the cell after.

**Only the built-in camera shows up?** The receiver isn't being seen as a camera. Unplug it and plug it back in (straight into the Mac, not through a hub), close any other app using a camera, and run the cell again. If it's still missing, run `sudo killall VDCAssistant AppleCameraAssistant` in a terminal and try again.

In [ ]:
import cv2
import matplotlib.pyplot as plt

max_index = 5
found = []

for index in range(max_index):
    cap = cv2.VideoCapture(index)
    if not cap.isOpened():
        print(f'{index}: not available')
        continue

    # The first few frames from a just-opened camera can be black, so read a few.
    ret, frame = False, None
    for _ in range(10):
        ret, frame = cap.read()
    cap.release()

    if not ret or frame is None:
        print(f'{index}: opened, but no frame')
        continue

    print(f'{index}: OK, frame captured ({frame.shape[1]}x{frame.shape[0]})')
    found.append((index, frame))

if found:
    fig, axes = plt.subplots(1, len(found), figsize=(4 * len(found), 3), squeeze=False)
    for ax, (index, frame) in zip(axes[0], found):
        ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        ax.set_title(f'camera = {index}')
        ax.axis('off')
    plt.show()
else:
    print('No cameras found -- check the receiver is plugged in and camera permission is granted.')

In [ ]:
# Set this to the receiver's index from the scan above. Pass the same number to
# keyboard_control_IL.py with --camera.
camera = 0

## Part 2: Live View

A window showing the live feed (or static, until the receiver locks on) should appear. **Leave it open, power on the drone, and hold the receiver's left button** for a few seconds so it scans and locks onto your camera (steps 3-5 in Part 0). Wave a hand in front of the camera to check that the picture is yours. Press `q` in the window to close it. The cell then prints the frame size and the frame rate it actually measured.

In [ ]:
import time

cap = cv2.VideoCapture(camera)

if not cap.isOpened():
    print(f'Could not open camera {camera}. Re-run the scan in Part 1 and pick a different index.')
else:
    n_frames = 0
    t_start = time.time()
    frame_shape = None

    while True:
        # Capture frame-by-frame
        ret, frame = cap.read()
        if not ret or frame is None:
            print('Failed to read a frame -- check the receiver is plugged in, or try a different `camera` value.')
            break
        n_frames += 1
        frame_shape = frame.shape

        # Display the resulting frame
        cv2.imshow('frame', frame)

        # Hit q to quit.
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    elapsed = time.time() - t_start

    # Release the capture
    cap.release()
    cv2.destroyAllWindows()
    # On some Macs, destroyAllWindows() alone can leave the window frozen; pumping a
    # few more waitKey calls flushes the GUI event queue so it actually closes.
    for _ in range(5):
        cv2.waitKey(1)

    if frame_shape is not None:
        print(f'Frame size: {frame_shape[1]}x{frame_shape[0]}')
        print(f'Measured frame rate: {n_frames / elapsed:.1f} fps')

## Part 3: Crop Check

The receiver draws an overlay along the top of the image: signal strength, frequency, signal strength (e.g. `92%  5789MHz  94%`). `keyboard_control_IL.py` cuts off the top `CROP_SIZE` rows of every frame before saving it, so the overlay doesn't end up in the training images. **The default is 80**, and the cell below draws it as a red line.

The receiver doesn't always show the overlay. When it does, the text can reach down to about row 105, below the 80 line. If you see text below the red line, record with `--crop 110` instead. The dataloader brings all trials to the same crop, and training and flying use that same crop (see `README.md`). Don't pick any other number.

In [ ]:
CROP_SIZE = 80  # pass this to keyboard_control_IL.py with --crop

cap = cv2.VideoCapture(camera)
ret, frame = False, None
for _ in range(10):
    ret, frame = cap.read()
cap.release()

if not ret or frame is None:
    print('Could not read a frame -- check Part 2 first.')
else:
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    cropped = rgb[CROP_SIZE:, :, :]

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].imshow(rgb)
    axes[0].axhline(CROP_SIZE, color='red', linewidth=2)
    axes[0].set_title(f'Full frame {frame.shape[1]}x{frame.shape[0]} (red = crop line)')
    axes[1].imshow(cropped)
    axes[1].set_title(f'Saved image {cropped.shape[1]}x{cropped.shape[0]}')
    for ax in axes:
        ax.axis('off')
    plt.show()

## Part 4: Record and Play Back

The following cell will connect to the camera module on the crazyflie and record a 5-second video. The cell will tell you when the video recording starts and ends. Move the drone by hand during the recording (tilt it, move it forward). The result is a video titled `video_crazyflie.avi`.

The video is saved at the frame rate actually measured during the recording, so it plays back at real speed.

In [ ]:
record_seconds = 5
file_name = 'video_crazyflie_oldcamera.avi'

cap = cv2.VideoCapture(camera)

frames = []
print('Recording video!')
t_start = time.time()
while time.time() - t_start < record_seconds:
    # Capture frame-by-frame
    ret, frame = cap.read()
    if ret and frame is not None:
        frames.append(frame)
elapsed = time.time() - t_start
print('Recording finished!')

# Release the capture
cap.release()

if len(frames) == 0:
    print('No frames were captured -- check Part 2 first.')
else:
    fps = len(frames) / elapsed
    image_size = (frames[0].shape[1], frames[0].shape[0])  # (width, height)
    video = cv2.VideoWriter(file_name, cv2.VideoWriter_fourcc('M', 'J', 'P', 'G'), fps, image_size)
    for frame in frames:
        video.write(frame)
    video.release()
    print(f'Saved {file_name}: {len(frames)} frames, {image_size[0]}x{image_size[1]}, {fps:.1f} fps')

Next, load the video back from disk and turn it into a GIF (saved as `anim.gif` next to this notebook). If it plays back what you recorded, the camera side works.

In [ ]:
import numpy as np
from matplotlib import animation
from IPython.display import Image, display

def load_video(name):
    frames = []
    cap = cv2.VideoCapture(name)
    fps = cap.get(cv2.CAP_PROP_FPS)
    while cap.isOpened():
        ret, frame = cap.read()

        if ret == False:
            break
        else:
            frames.append(frame)

    cap.release()

    height, width = frames[0].shape[:2]
    I = np.zeros([height, width, len(frames)])
    for t in range(len(frames)):
        I[:, :, t] = cv2.cvtColor(frames[t], cv2.COLOR_BGR2GRAY) / 255

    return I, fps

def create_gif(I, frame_rate=20):
    fig, ax = plt.subplots()
    im = ax.imshow(I[:, :, 0], cmap='gray')
    ax.axis('off')

    def animate(i):
        im.set_data(I[:, :, i])
        return [im]

    anim = animation.FuncAnimation(fig, animate, frames=I.shape[2], interval=1000 / frame_rate, blit=True)
    anim.save('./anim.gif', writer='pillow', fps=frame_rate)
    plt.close(fig)

print('Loading video!')
I, frame_rate = load_video(file_name)
print(f'{I.shape[2]} frames at {frame_rate:.1f} fps')

print('Making GIF!')
create_gif(I, frame_rate)
print('GIF made!')
display(Image(filename='anim.gif'))

## Part 5: Flight Test (Takeoff, Hover, Land)

A basic takeoff, hover and land. Run it with the camera mounted, to check that the drone still flies with the extra weight.

**The F2026 drone is a Crazyflie 2.1 Brushless, which must be armed before the motors will spin:** `cf.platform.send_arming_request(True)`. No error is raised if you skip it; the drone just sits there. The drone must be flat, still, and props clear when it's sent. `keyboard_control_IL.py` does the same thing when you press `u`.

Give the drone at least a 1m x 1m clear, flat area.

In [ ]:
drone_number = 1  # set to your drone's radio channel number

In [ ]:
# Takeoff/landing maneuvers borrowed from Lab2-sol.ipynb's hardware implementation
# (ascend_and_hover / hover_and_descend), with the RRT waypoint-following step left out.

import time
import cflib.crtp
from cflib.crazyflie import Crazyflie
from cflib.crazyflie.log import LogConfig
from cflib.crazyflie.syncCrazyflie import SyncCrazyflie
from cflib.crazyflie.syncLogger import SyncLogger

# Determine initial position:
def wait_for_position_estimator(scf):
    print('Waiting for estimator to find position...')

    log_config = LogConfig(name='Kalman Variance', period_in_ms=500)
    log_config.add_variable('kalman.varPX', 'float')
    log_config.add_variable('kalman.varPY', 'float')
    log_config.add_variable('kalman.varPZ', 'float')

    var_y_history = [1000] * 10
    var_x_history = [1000] * 10
    var_z_history = [1000] * 10

    threshold = 0.001
    with SyncLogger(scf, log_config) as logger:
        for log_entry in logger:
            data = log_entry[1]

            var_x_history.append(data['kalman.varPX'])
            var_x_history.pop(0)
            var_y_history.append(data['kalman.varPY'])
            var_y_history.pop(0)
            var_z_history.append(data['kalman.varPZ'])
            var_z_history.pop(0)

            min_x = min(var_x_history)
            max_x = max(var_x_history)
            min_y = min(var_y_history)
            max_y = max(var_y_history)
            min_z = min(var_z_history)
            max_z = max(var_z_history)

            print("{} {} {}".
                format(max_x - min_x, max_y - min_y, max_z - min_z))

            if (max_x - min_x) < threshold and (
                    max_y - min_y) < threshold and (
                    max_z - min_z) < threshold:
                break

# Initialize controller:
def set_PID_controller(cf):
    # Set the PID Controller:
    print('Initializing PID Controller')
    cf.param.set_value('stabilizer.controller', '1')
    cf.param.set_value('kalman.resetEstimation', '1')
    time.sleep(0.1)
    cf.param.set_value('kalman.resetEstimation', '0')

    wait_for_position_estimator(cf)
    time.sleep(0.1)
    return

# Ascend and hover:
def ascend_and_hover(cf):
    # Ascend:
    for y in range(20):
        cf.commander.send_hover_setpoint(0, 0, 0, y / 50)
        time.sleep(0.1)
    # Hover at 0.5 meters:
    for _ in range(30):
        cf.commander.send_hover_setpoint(0, 0, 0, 0.5)
        time.sleep(0.1)
    return

# Hover, descend, and stop all motion:
def hover_and_descend(cf):
    # Hover at 0.5 meters:
    for _ in range(30):
        cf.commander.send_hover_setpoint(0, 0, 0, 0.5)
        time.sleep(0.1)
    # Descend:
    for y in range(10):
        cf.commander.send_hover_setpoint(0, 0, 0, (10 - y) / 25)
        time.sleep(0.1)
    # Stop all motion:
    for i in range(10):
        cf.commander.send_stop_setpoint()
        time.sleep(0.1)
    return

def run_flight_test(drone_number):
    # Sanity-check flight: takeoff, hover, land. No waypoints, no camera.
    uri = f'radio://0/{drone_number}/2M'

    cflib.crtp.init_drivers(enable_debug_driver=False)

    with SyncCrazyflie(uri, cf=Crazyflie(rw_cache='./cache')) as scf:
        cf = scf.cf

        set_PID_controller(cf)

        # Mandatory on the Crazyflie 2.1 Brushless: the motors will not spin
        # without this. Drone must be flat, still, and props clear when sent.
        cf.platform.send_arming_request(True)
        time.sleep(1.0)

        print('Taking off!')
        ascend_and_hover(cf)

        print('Landing!')
        hover_and_descend(cf)

        cf.platform.send_arming_request(False)

    print('Done!')
    return

In [ ]:
run_flight_test(drone_number)

## Part 6: Fly While Recording

This runs Part 5's flight and Part 4's recording **at the same time**: one thread flies the drone while the main thread records from the receiver, until the flight finishes plus 2 extra seconds to catch the landing. It checks that the 5.8 GHz video link and the 2.4 GHz Crazyradio don't interfere, and that the picture holds up with the motors running (vibration, motor noise on the camera's power). It is also what the IL script does: fly and record at once.

The result is saved as `flight_with_recording.avi`.

In [ ]:
import threading
import os

flight_done = threading.Event()
flight_error = []

def _fly():
    try:
        run_flight_test(drone_number)
    except Exception as e:
        flight_error.append(e)
    finally:
        flight_done.set()

cap = cv2.VideoCapture(camera)
if not cap.isOpened():
    print(f'Could not open camera {camera} -- fix Part 2 before flying.')
else:
    flight_thread = threading.Thread(target=_fly)
    flight_thread.start()

    frames = []
    dropped_frames = 0
    print('Recording while the drone flies...')
    t_start = time.time()

    # Keep recording until the flight thread signals it's done, plus a couple
    # extra seconds to catch the very end of landing/disarm.
    t_stop = None
    while t_stop is None or time.time() < t_stop:
        if t_stop is None and flight_done.is_set():
            t_stop = time.time() + 2.0
        ret, frame = cap.read()
        if ret and frame is not None:
            frames.append(frame)
        else:
            dropped_frames += 1
    elapsed = time.time() - t_start

    cap.release()
    flight_thread.join()
    print('Recording finished!')

    if dropped_frames > 0:
        print(f'Warning: {dropped_frames} frame(s) failed to read during recording.')

    if len(frames) > 0:
        file_name = 'flight_with_recording.avi'
        fps = len(frames) / elapsed
        image_size = (frames[0].shape[1], frames[0].shape[0])
        video = cv2.VideoWriter(file_name, cv2.VideoWriter_fourcc('M', 'J', 'P', 'G'), fps, image_size)
        for frame in frames:
            video.write(frame)
        video.release()
        print(f'Saved {file_name}: {len(frames)} frames, {fps:.1f} fps')

    if flight_error:
        raise flight_error[0]

### Convert the flight video to a GIF

This turns `flight_with_recording.avi` into a color GIF (`flight_with_recording.gif`, next to this notebook) that plays at real speed and is easy to share or drop into slides. Each frame is shrunk to 320 pixels wide and every 2nd frame is kept, which puts a ~20 s flight at about 15 MB. Lower `width`, or raise `frame_step` to 3, for a smaller file. Change `video_name` to convert any other `.avi`, e.g. Part 4's recording.

Color smearing or tearing in the GIF (green/purple bands) is in the original video too: it's analog signal breakup, often worst near landing.

In [ ]:
import os
import cv2
from PIL import Image as PILImage
from IPython.display import Image, display

video_name = 'flight_with_recording.avi'
gif_name = 'flight_with_recording.gif'

def avi_to_gif(video_name, gif_name, width=320, frame_step=2):
    # Color GIF of a recorded video. Shrinks each frame to `width` pixels wide and keeps
    # every `frame_step`-th frame, so a ~20 s flight stays a reasonable file size.
    cap = cv2.VideoCapture(video_name)
    fps = cap.get(cv2.CAP_PROP_FPS)

    frames = []
    index = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if index % frame_step == 0:
            height = int(frame.shape[0] * width / frame.shape[1])
            small = cv2.resize(frame, (width, height), interpolation=cv2.INTER_AREA)
            rgb = cv2.cvtColor(small, cv2.COLOR_BGR2RGB)
            frames.append(PILImage.fromarray(rgb))
        index += 1
    cap.release()

    if len(frames) == 0:
        print(f'Could not read any frames from {video_name}')
        return False

    # Same playback speed as the video: each GIF frame lasts frame_step video frames
    gif_fps = fps / frame_step
    frame_ms = int(round(1000 / gif_fps))
    frames[0].save(gif_name, save_all=True, append_images=frames[1:], duration=frame_ms, loop=0)

    size_mb = os.path.getsize(gif_name) / 1e6
    print(f'Saved {gif_name}: {len(frames)} frames, {frames[0].width}x{frames[0].height}, {gif_fps:.1f} fps, {size_mb:.1f} MB')
    return True

if avi_to_gif(video_name, gif_name):
    display(Image(filename=gif_name))

## Next: Collect Trajectories

Once Parts 0-3 work (4-6 are optional), collect imitation-learning data with `keyboard_control_IL.py`. It must run from a **terminal**, not from this notebook (its keyboard listener doesn't work inside Jupyter). Close this notebook's kernel first so the camera and radio are free. See `README.md` for the full steps:

```
conda activate mae345
cd <this folder>
python keyboard_control_IL.py --group <radio channel> --camera <camera>                # add --crop 110 if Part 3 showed overlay below the line
python keyboard_control_IL.py --group <radio channel> --camera <camera> --out <folder>  # save to a folder other than imitation_data/
python analyze_il_data.py imitation_data/trial_01
```